In [29]:
import sc2reader
import pandas as pd
import numpy as np
import tqdm
from glob import glob
from utils import *

all_dfs = []
game_number = 0
for account in ['sample']:
    files = glob(f'../data/input/my_data/{account}/*')
    for file in tqdm.tqdm(files):
        
        try:
            replay = sc2reader.load_replay(file, load_map=True)
        except Exception as e:
            print(f"Failed to load {file}: {e}")
            continue
        if replay.map_name in ['Ruby Rock LE', 'Emerald City CE', 'Reclamation LE', 'Fields of Death', 'Gemgarden LE', 'New Bed of Chaos LE', 'Rhoskallian LE', 'Rust Bucket LE', 'Sludge City', 'Undercurrent LE', 'Yellowjacket', 'Phantom Mode']:
            continue
        game_number += 1
        assert replay.map_name in ['valid_maps', "At Eternity's Edge LE", 'Blackrock LE', 'Fear and Faith LE', 'Rainfall LE', 'Sanctuary III LE', 'Lockdown LE', 'Washout LE', 'Rorschach LE', 'Old Sun Temple LE'], replay.map_name

        is_valid_release = replay.release_string >= '5.0.16'
        assert is_valid_release, replay.release_string
        
        player1 = 'nemo'
        player2 = None
        player1_won = None
        player1_race = 'Zerg'
        player2_race = None
        for player in replay.players:
            if player.name == 'nemo' or player.name == 'Kairo':
                assert player.play_race == 'Zerg'
                player1_won = player.result
            else:
                if player2 != None:
                    raise Exception(player.name, player2)
                player2 = player.name
                player2_race = player.play_race
        print(player1, player2, player1_race, player2_race)

        inject_times_data = []
        for event in replay.events:
            seconds = event.frame / 22.4
            minutes = int(seconds // 60)
            remaining_seconds = int(seconds % 60)
            if seconds < 210:
                continue
        
            try:
                if player2 in str(event):
                    continue
                if event.player == player2:
                    continue
            except:
                pass

            print(seconds, event)
            if event.name == 'TargetUnitCommandEvent':
                print(seconds, event)
                if event.ability == None:
                    continue
                if event.ability.name == 'SpawnLarva':
                    inject_times_data.append(seconds)
            if seconds > 10*60:
                break
        inject_times_df = pd.DataFrame()
        inject_times_df['inject_time'] = inject_times_data
        inject_times_df['game_number'] = game_number
        inject_times_df['opponent_race'] = player2_race
        inject_times_df['game_length_seconds_max_600'] = int(np.round(seconds,0))
        inject_times_df['main_player_won'] = player1_won
        inject_times_df = inject_times_df[['game_number', 'opponent_race', 'game_length_seconds_max_600', 'inject_time', 'main_player_won']]
        all_dfs.append(inject_times_df)
inject_times_df = pd.concat(all_dfs, axis='index', ignore_index=True)
# local.write.csv(inject_times_df, '3_extract_my_injects')

 33%|███▎      | 1/3 [00:01<00:03,  1.54s/it]

nemo Mandus Zerg Zerg
18.571428571428573 00.26	nemo            Right Click; Target: LabMineralField [02400001]; Location: (148.0, 30.5, 49120)
22.99107142857143 00.32	nemo            Right Click; Target: LabMineralField [00600001]; Location: (152.0, 34.5, 49120)
28.34821428571429 00.39	nemo            Ability (1722) - BuildExtractor; Target: SpacePlatformGeyser [02040001]; Location: (152.5, 39.5, 49120)
101.96428571428572 02.22	nemo            Ability (1722) - BuildExtractor; Target: SpacePlatformGeyser [02040001]; Location: (152.5, 39.5, 49120)
105.49107142857143 02.27	nemo            Right Click; Target: LabMineralField750 [01AC0001]; Location: (152.0, 32.5, 49120)
111.91964285714286 02.36	nemo            Right Click; Target: Extractor [03880001]; Location: (152.5, 39.5, 49120)
120.84821428571429 02.49	nemo            Right Click; Target: Extractor [03880001]; Location: (152.5, 39.5, 49120)
182.94642857142858 04.16	nemo            Ability (E20) - SpawnLarva; Target: Lair [02F40001]; 

 67%|██████▋   | 2/3 [00:02<00:01,  1.17s/it]

nemo Red Zerg Terran
9.285714285714286 00.13	nemo            Right Click; Target: MineralField [01A80001]; Location: (178.0, 67.5, 57296)
13.705357142857144 00.19	nemo            Right Click; Target: MineralField [01E00001]; Location: (185.0, 73.5, 57296)
27.05357142857143 00.37	nemo            Ability (1722) - BuildExtractor; Target: ShakurasVespeneGeyser [01200001]; Location: (185.5, 76.5, 57296)
48.79464285714286 01.08	nemo            Right Click; Target: MineralField [01E00001]; Location: (185.0, 73.5, 57296)
90.08928571428572 02.06	nemo            Right Click; Target: MineralField [01580001]; Location: (185.0, 71.5, 57296)
106.60714285714286 02.29	nemo            Ability (1722) - BuildExtractor; Target: ShakurasVespeneGeyser [01200001]; Location: (185.5, 76.5, 57296)
111.51785714285715 02.36	nemo            Right Click; Target: MineralField750 [01900001]; Location: (185.0, 69.5, 57296)
138.16964285714286 03.13	nemo            Ability (563) - ScanMove; Target: ShakurasVespeneGeyser

100%|██████████| 3/3 [00:03<00:00,  1.27s/it]

nemo TheReckoner Zerg Protoss
13.660714285714286 00.19	nemo            Right Click; Target: LabMineralField [02D80001]; Location: (46.0, 22.5, 57280)
18.92857142857143 00.26	nemo            Right Click; Target: LabMineralField [00C00001]; Location: (43.0, 22.5, 57280)
22.99107142857143 00.32	nemo            Right Click; Target: LabMineralField [00300001]; Location: (39.0, 26.5, 57280)
26.651785714285715 00.37	nemo            Ability (1722) - BuildExtractor; Target: SpacePlatformGeyser [02680001]; Location: (38.5, 31.5, 57280)
32.05357142857143 00.44	nemo            Right Click; Target: Drone [03B00001]; Location: (40.35498046875, 28.465576171875, 57280)
91.83035714285715 02.08	nemo            Right Click; Target: Drone [039C0001]; Location: (42.129638671875, 27.4619140625, 57280)
104.46428571428572 02.26	nemo            Ability (1722) - BuildExtractor; Target: SpacePlatformGeyser [026C0001]; Location: (49.5, 21.5, 57280)
111.91964285714286 02.36	nemo            Right Click; Target: Lab

In [30]:
output_df



,unspent_time,unspent_amount,game_number
0,300.000000,300,1
1,307.142857,220,1
2,314.285714,235,1
3,321.428571,262,1
4,328.571429,238,1
5,335.714286,158,1
6,342.857143,170,1
7,350.000000,140,1
